In [ ]:
from google.colab import files
uploaded = files.upload()


Saving dynamic_pricing.csv to dynamic_pricing.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Set the working directory and load data
ride_data = pd.read_csv('dynamic_pricing.csv')

# Split the data into training and testing sets
train_data, test_data = train_test_split(ride_data, test_size=0.2, random_state=123)

# Check for missing values
print(train_data.isnull().sum())
print(test_data.isnull().sum())

# Check for duplicate rows
duplicates_train = train_data[train_data.duplicated()]
print(duplicates_train)

duplicates_test = test_data[test_data.duplicated()]
print(duplicates_test)

# Calculate demand_multiplier based on percentiles
high_demand_percentile = 75
low_demand_percentile = 25

high_demand_value = np.percentile(train_data['Number_of_Riders'], high_demand_percentile)
low_demand_value = np.percentile(train_data['Number_of_Riders'], low_demand_percentile)



Number_of_Riders           0
Number_of_Drivers          0
Location_Category          0
Customer_Loyalty_Status    0
Number_of_Past_Rides       0
Average_Ratings            0
Time_of_Booking            0
Vehicle_Type               0
Expected_Ride_Duration     0
Historical_Cost_of_Ride    0
dtype: int64
Number_of_Riders           0
Number_of_Drivers          0
Location_Category          0
Customer_Loyalty_Status    0
Number_of_Past_Rides       0
Average_Ratings            0
Time_of_Booking            0
Vehicle_Type               0
Expected_Ride_Duration     0
Historical_Cost_of_Ride    0
dtype: int64
Empty DataFrame
Columns: [Number_of_Riders, Number_of_Drivers, Location_Category, Customer_Loyalty_Status, Number_of_Past_Rides, Average_Ratings, Time_of_Booking, Vehicle_Type, Expected_Ride_Duration, Historical_Cost_of_Ride]
Index: []
Empty DataFrame
Columns: [Number_of_Riders, Number_of_Drivers, Location_Category, Customer_Loyalty_Status, Number_of_Past_Rides, Average_Ratings, Time_of_Book

In [ ]:
train_data['demand_multiplier'] = np.where(
    train_data['Number_of_Riders'] > high_demand_value,
    train_data['Number_of_Riders'] / high_demand_value,
    train_data['Number_of_Riders'] / low_demand_value
)

test_data['demand_multiplier'] = np.where(
    test_data['Number_of_Riders'] > high_demand_value,
    test_data['Number_of_Riders'] / high_demand_value,
    test_data['Number_of_Riders'] / low_demand_value
)

# Calculate supply_multiplier based on percentiles
high_supply_percentile = 75
low_supply_percentile = 25

high_supply_value = np.percentile(train_data['Number_of_Drivers'], high_supply_percentile)
low_supply_value = np.percentile(train_data['Number_of_Drivers'], low_supply_percentile)

train_data['supply_multiplier'] = np.where(
    train_data['Number_of_Drivers'] > low_supply_value,
    high_supply_value / train_data['Number_of_Drivers'],
    low_supply_value / train_data['Number_of_Drivers']
)

test_data['supply_multiplier'] = np.where(
    test_data['Number_of_Drivers'] > low_supply_value,
    high_supply_value / test_data['Number_of_Drivers'],
    low_supply_value / test_data['Number_of_Drivers']
)

# Define price adjustment thresholds
demand_threshold_high = 1.2
demand_threshold_low = 0.8
supply_threshold_high = 0.8
supply_threshold_low = 1.2

# Calculate adjusted_ride_cost
train_data['adjusted_ride_cost'] = train_data['Historical_Cost_of_Ride'] * (
    np.maximum(train_data['demand_multiplier'], demand_threshold_low) *
    np.maximum(train_data['supply_multiplier'], supply_threshold_high)
)

test_data['adjusted_ride_cost'] = test_data['Historical_Cost_of_Ride'] * (
    np.maximum(test_data['demand_multiplier'], demand_threshold_low) *
    np.maximum(test_data['supply_multiplier'], supply_threshold_high)
)


In [ ]:
train_data['demand_multiplier'] = np.where(
    train_data['Number_of_Riders'] > high_demand_value,
    train_data['Number_of_Riders'] / high_demand_value,
    train_data['Number_of_Riders'] / low_demand_value
)

test_data['demand_multiplier'] = np.where(
    test_data['Number_of_Riders'] > high_demand_value,
    test_data['Number_of_Riders'] / high_demand_value,
    test_data['Number_of_Riders'] / low_demand_value
)

# Calculate supply_multiplier based on percentiles
high_supply_percentile = 75
low_supply_percentile = 25

high_supply_value = np.percentile(train_data['Number_of_Drivers'], high_supply_percentile)
low_supply_value = np.percentile(train_data['Number_of_Drivers'], low_supply_percentile)

train_data['supply_multiplier'] = np.where(
    train_data['Number_of_Drivers'] > low_supply_value,
    high_supply_value / train_data['Number_of_Drivers'],
    low_supply_value / train_data['Number_of_Drivers']
)

test_data['supply_multiplier'] = np.where(
    test_data['Number_of_Drivers'] > low_supply_value,
    high_supply_value / test_data['Number_of_Drivers'],
    low_supply_value / test_data['Number_of_Drivers']
)

# Define price adjustment thresholds
demand_threshold_high = 1.2
demand_threshold_low = 0.8
supply_threshold_high = 0.8
supply_threshold_low = 1.2

# Calculate adjusted_ride_cost
train_data['adjusted_ride_cost'] = train_data['Historical_Cost_of_Ride'] * (
    np.maximum(train_data['demand_multiplier'], demand_threshold_low) *
    np.maximum(train_data['supply_multiplier'], supply_threshold_high)
)

test_data['adjusted_ride_cost'] = test_data['Historical_Cost_of_Ride'] * (
    np.maximum(test_data['demand_multiplier'], demand_threshold_low) *
    np.maximum(test_data['supply_multiplier'], supply_threshold_high)
)


In [ ]:
# Prepare train and test sets
drop_cols = ['demand_multiplier', 'supply_multiplier', 'Historical_Cost_of_Ride']
train_set = train_data.drop(columns=drop_cols)
test_set = test_data.drop(columns=drop_cols)

# Identify categorical variables
cat_vars = train_set.select_dtypes(include=['object', 'category']).columns.tolist()

# One-hot encoding using OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit encoder on training set categorical variables
encoder.fit(train_set[cat_vars])

# Transform both training and test sets
train_cat_encoded = encoder.transform(train_set[cat_vars])
test_cat_encoded = encoder.transform(test_set[cat_vars])

# Convert to DataFrames and set proper column names
encoded_cols = encoder.get_feature_names_out(cat_vars)
train_cat_df = pd.DataFrame(train_cat_encoded, columns=encoded_cols, index=train_set.index)
test_cat_df = pd.DataFrame(test_cat_encoded, columns=encoded_cols, index=test_set.index)

# Drop original categorical columns and add encoded ones
train_encoded = pd.concat([train_set.drop(columns=cat_vars), train_cat_df], axis=1)
test_encoded = pd.concat([test_set.drop(columns=cat_vars), test_cat_df], axis=1)

# Standard scaling for numerical variables
numerical_vars = [col for col in train_encoded.columns if col not in encoded_cols and col != 'adjusted_ride_cost']

scaler = StandardScaler()
train_encoded[numerical_vars] = scaler.fit_transform(train_encoded[numerical_vars])
test_encoded[numerical_vars] = scaler.transform(test_encoded[numerical_vars])

# Final train_encoded and test_encoded datasets are ready
print(train_encoded.head())
print(test_encoded.head())

     Number_of_Riders  Number_of_Drivers  Number_of_Past_Rides  \
512         -1.658856          -1.090327             -1.219616   
685         -0.005353          -0.569574              0.610045   
997         -0.683713          -1.090327              1.024308   
927         -1.616459          -0.882026             -1.564835   
376         -1.489266          -0.986176             -0.701787   

     Average_Ratings  Expected_Ride_Duration  adjusted_ride_cost  \
512        -0.999101               -0.393433          467.634205   
685        -0.032285                0.182160         1629.765778   
997        -0.262479               -1.215709          325.489649   
927         0.451123                0.613854          415.168250   
376         0.451123                0.326058          473.864436   

     Location_Category_Suburban  Location_Category_Urban  \
512                         1.0                      0.0   
685                         0.0                      1.0   
997           

In [ ]:
# Total NaN values per column
print("Missing values in train_encoded:")
print(train_encoded.isna().sum())

print("\nMissing values in test_encoded:")
print(test_encoded.isna().sum())

# Optionally, total number of missing values overall
print("\nTotal missing values in train_encoded:", train_encoded.isna().sum().sum())
print("Total missing values in test_encoded:", test_encoded.isna().sum().sum())


Missing values in train_encoded:
Number_of_Riders                   0
Number_of_Drivers                  0
Number_of_Past_Rides               0
Average_Ratings                    0
Expected_Ride_Duration             0
adjusted_ride_cost                 0
Location_Category_Suburban         0
Location_Category_Urban            0
Customer_Loyalty_Status_Regular    0
Customer_Loyalty_Status_Silver     0
Time_of_Booking_Evening            0
Time_of_Booking_Morning            0
Time_of_Booking_Night              0
Vehicle_Type_Premium               0
dtype: int64

Missing values in test_encoded:
Number_of_Riders                   0
Number_of_Drivers                  0
Number_of_Past_Rides               0
Average_Ratings                    0
Expected_Ride_Duration             0
adjusted_ride_cost                 0
Location_Category_Suburban         0
Location_Category_Urban            0
Customer_Loyalty_Status_Regular    0
Customer_Loyalty_Status_Silver     0
Time_of_Booking_Evening          

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

# Separate features and target
X_train = train_encoded.drop(columns=['adjusted_ride_cost'])
y_train = train_encoded['adjusted_ride_cost']

X_test = test_encoded.drop(columns=['adjusted_ride_cost'])
y_test = test_encoded['adjusted_ride_cost']

# Define parameter grid to reduce overfitting
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 0.5]
}

# Grid search
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=123, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
grid_search.fit(X_train, y_train)

# Use best estimator
best_rf_model = grid_search.best_estimator_

# Predict on both train and test
y_predt = best_rf_model.predict(X_train)
y_pred = best_rf_model.predict(X_test)

# Evaluate
import numpy as np

rmset = np.sqrt(mean_squared_error(y_train, y_predt))
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2t = r2_score(y_train, y_predt)
r2 = r2_score(y_test, y_pred)

# Print results
print("Best parameters from tuning:", grid_search.best_params_)
print(f"\nRandom Forest RMSE train: {rmset:.2f}")
print(f"Random Forest R² train: {r2t:.4f}")
print(f"Random Forest RMSE test: {rmse:.2f}")
print(f"Random Forest R² test: {r2:.4f}")



Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters from tuning: {'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

Random Forest RMSE train: 116.72
Random Forest R² train: 0.9435
Random Forest RMSE test: 180.20
Random Forest R² test: 0.8312
